In [1]:
! pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 10.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.1/124.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.9/246.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.9 MB/s eta 0:00:00:00:01


In [2]:
from fastai.vision.all import *
from fastbook import *

In [3]:
path = untar_data(URLs.PASCAL_2007)
path

<div><progress max="1637796771" value="1637801984"></progress> 100.00% [1637801984/1637796771 00:37&lt;00:00]</div>

Path('/root/.fastai/data/pascal_2007')

In [4]:
Path.BASE_PATH = path
path

Path('.')

In [5]:
path.ls()

[Path('test.json'), Path('train.csv'), Path('test'), Path('valid.json'), Path('train'), Path('test.csv'), Path('segmentation'), Path('train.json')]

In [17]:
(path/'train').ls()

(#5012) [Path('train/009528.jpg'), Path('train/009721.jpg'), Path('train/006269.jpg'), Path('train/007762.jpg'), Path('train/001586.jpg'), Path('train/004846.jpg'), Path('train/006628.jpg'), Path('train/007899.jpg'), Path('train/001981.jpg'), Path('train/008722.jpg'), Path('train/000030.jpg'), Path('train/004955.jpg'), Path('train/005421.jpg'), Path('train/005230.jpg'), Path('train/007330.jpg'), Path('train/002411.jpg'), Path('train/003658.jpg'), Path('train/002657.jpg'), Path('train/008398.jpg'), Path('train/000777.jpg'), Path('train/004244.jpg'), Path('train/003063.jpg'), Path('train/000009.jpg'), Path('train/009809.jpg'), Path('train/001231.jpg'), Path('train/009654.jpg'), Path('train/006802.jpg'), Path('train/009792.jpg'), Path('train/004185.jpg'), Path('train/008468.jpg'), Path('train/001517.jpg'), Path('train/001521.jpg'), Path('train/004196.jpg'), Path('train/007606.jpg'), Path('train/006983.jpg'), Path('train/009022.jpg'), Path('train/009443.jpg'), Path('train/003859.jpg'), Pat

In [6]:
df = pd.read_csv(path/'train.csv')
df.head()

,fname,labels,is_valid
0,000005.jpg,chair,True
1,000007.jpg,car,True
2,000009.jpg,horse person,True
3,000012.jpg,car,False
4,000016.jpg,bicycle,True


In [7]:
df.iloc[:,0]

0       000005.jpg
1       000007.jpg
2       000009.jpg
3       000012.jpg
4       000016.jpg
           ...    
5006    009954.jpg
5007    009955.jpg
5008    009958.jpg
5009    009959.jpg
5010    009961.jpg
Name: fname, Length: 5011, dtype: object

In [9]:
df.iloc[0,:]

fname       000005.jpg
labels           chair
is_valid          True
Name: 0, dtype: object

In [10]:
# same as above
df.iloc[0]

fname       000005.jpg
labels           chair
is_valid          True
Name: 0, dtype: object

In [11]:
# individual column values
df['is_valid']

0        True
1        True
2        True
3       False
4        True
        ...  
5006     True
5007     True
5008     True
5009    False
5010    False
Name: is_valid, Length: 5011, dtype: bool

In [13]:
# create columns and do calculations
tmp_df = pd.DataFrame({'a': [1,2], 'b': [3,4]})
tmp_df

,a,b
0,1,3
1,2,4


In [14]:
tmp_df['c'] = tmp_df['a'] + tmp_df['b']
tmp_df

,a,b,c
0,1,3,4
1,2,4,6


In [15]:
dblock = DataBlock()
dblock

In [22]:
dsets = dblock.datasets(df)
len(dsets.train), len(dsets.valid)

(4009, 1002)

In [26]:
x, y = dsets.train[0]
x, y

(fname       004347.jpg
 labels            bird
 is_valid         False
 Name: 2187, dtype: object,
 fname       004347.jpg
 labels            bird
 is_valid         False
 Name: 2187, dtype: object)

In [27]:
x['fname'], x['labels']

('004347.jpg', 'bird')

In [29]:
dblock = DataBlock(
    get_x=lambda x: x['fname'],
    get_y=lambda x: x['labels'].split(' '),
)
dsets = dblock.datasets(df)
dsets.train[0]

('005624.jpg', ['bird'])

In [33]:
# using full function definitions (preferable you need to export your learner because lambda functions can't be serialized)
def get_x(row):
    return row['fname']

def get_y(row):
    return row['labels'].split(' ')

dblock = DataBlock(get_y=get_y, get_x=get_x)
dsets = dblock.datasets(df)
dsets.train[0], dsets.valid[0]

(('006932.jpg', ['person']), ('001693.jpg', ['car']))

In [34]:
# need to make the image paths full so the images are accessible
def get_x(row):
    return path/'train'/row['fname']

dblock = DataBlock(get_x=get_x, get_y=get_y)
dsets = dblock.datasets(df)
dsets.train[0], dsets.valid[0]

((Path('train/009546.jpg'), ['sofa', 'person']),
 (Path('train/000159.jpg'), ['person', 'car']))

In [35]:
# now actually access the images
dblock = DataBlock(
    blocks=(ImageBlock, MultiCategoryBlock),
    get_y=get_y,
    get_x=get_x,
)
dsets = dblock.datasets(df)
dsets.train[0], dsets.valid[0]

((PILImage mode=RGB size=500x375,
  TensorMultiCategory([0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0.])),
 (PILImage mode=RGB size=500x375,
  TensorMultiCategory([0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])))

In [40]:
# let's identify what category indices represent
idxs = torch.where(dsets.train[0][1]==1.)
idxs

(TensorMultiCategory([ 6, 12, 14]),)

In [46]:
dsets.train.vocab

['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']

In [44]:
dsets.train.vocab[dsets.train[0][1]==1.]

['car', 'horse', 'person']

In [49]:
# now let's split the data
df.index

RangeIndex(start=0, stop=5011, step=1)

In [48]:
# now let's split the data
df.index[df['is_valid']]

Index([   0,    1,    2,    4,    6,    7,    8,   10,   12,   18,
       ...
       4992, 4994, 4995, 4997, 5002, 5003, 5005, 5006, 5007, 5008],
      dtype='int64', length=2510)

In [50]:
df.index[~df['is_valid']]

Index([   3,    5,    9,   11,   13,   14,   15,   16,   17,   20,
       ...
       4991, 4993, 4996, 4998, 4999, 5000, 5001, 5004, 5009, 5010],
      dtype='int64', length=2501)

In [54]:
df.iloc[df.index[~df['is_valid']]]

,fname,labels,is_valid
3,000012.jpg,car,False
5,000017.jpg,person horse,False
9,000023.jpg,bicycle person,False
11,000026.jpg,car,False
13,000032.jpg,aeroplane person,False
...,...,...,...
5000,009944.jpg,motorbike person,False
5001,009945.jpg,sheep,False
5004,009949.jpg,sofa chair person,False
5009,009959.jpg,car,False


In [55]:
df.iloc[df.index[df['is_valid']]]

,fname,labels,is_valid
0,000005.jpg,chair,True
1,000007.jpg,car,True
2,000009.jpg,horse person,True
4,000016.jpg,bicycle,True
6,000019.jpg,cat,True
...,...,...,...
5003,009947.jpg,boat person,True
5005,009950.jpg,train person,True
5006,009954.jpg,horse person,True
5007,009955.jpg,boat,True


In [58]:
def splitter(d_f):
    train = d_f.index[~d_f['is_valid']]
    valid = d_f.index[d_f['is_valid']]

    return train, valid

In [59]:
dblock = DataBlock(
    blocks=(ImageBlock, MultiCategoryBlock),
    get_y=get_y,
    get_x=get_x,
    splitter=splitter
)
dsets = dblock.datasets(df)
len(dsets.train), len(dsets.valid)

(2501, 2510)